# Limitaciones Metodológicas y de los Datos



## Objetivo de la Etapa

In [ ]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source
from src.survey import kish_design_effect
from src.territory import compute_departmental_indicators

set_style()

df_h = pd.read_parquet(PATHS["analytical"] / "df_hogar_analytic.parquet")
df_p = pd.read_parquet(PATHS["analytical"] / "df_persona_analytic.parquet")
df_d = compute_departmental_indicators(df_h, df_p)

## 1. Diseño de Corte Transversal y Restricciones de Causalidad

### Naturaleza de los Datos

### Implicaciones Metodológicas

## 2. Aproximación de la Dimensión Económica: Autopercepción y Pobreza Imputada

### Variables Utilizadas

### Limitaciones Identificadas

## 3. Métricas Binarias de Acceso vs. Calidad Efectiva del Servicio

### La Trampa del "Sí/No"

In [ ]:
# Evidencia de la divergencia entre acceso binario y calidad tecnológica
movil_nat = (df_h["tiene_internet_movil"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
fijo_nat = (df_h["tiene_internet_fijo"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
comp_nat = (df_h["tiene_computador"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100

df_calidad = pd.DataFrame({
 "Dimensión Tecnológica": [
 "Acceso Móvil (Teléfono celular con datos)",
 "Equipamiento (Computador / Portátil en vivienda)",
 "Infraestructura Fija (Banda ancha residencial)"
 ],
 "Penetración Nacional (%)": [f"{movil_nat:.1f}%", f"{comp_nat:.1f}%", f"{fijo_nat:.1f}%"],
 "Limitación del Indicador en la Encuesta": [
 "No mide velocidad (Mbps), cupo de datos (prepago vs. postpago) ni estabilidad del enlace",
 "No captura antigüedad, capacidad de procesamiento ni obsolescencia del equipo",
 "No distingue entre tecnologías (fibra óptica FTTH, cable coaxial HFC o cobre ADSL)"
 ]
})

print("Tabla de Dimensiones de Acceso vs. Limitaciones de Calidad no Observables")
print("=" * 85)
display(df_calidad.to_string(index=False))

## 4. Complejidad del Diseño Muestral y Corrección de Errores Estándar

### El Efecto de Diseño (DEFF)

In [ ]:
kish_res = kish_design_effect(df_h["fex_c"])
deff_val = kish_res["deff_kish"]
n_eff = kish_res["n_efectivo"]
inflation_factor = np.sqrt(deff_val)
n_raw = len(df_h)

print("IMPACTO DEL DISEÑO COMPLEJO SOBRE LA INFERENCIA")
print("=" * 60)
print(f" Tamaño muestral bruto (n): {n_raw:,} hogares")
print(f" Efecto de diseño Kish (DEFF): {deff_val:.2f}")
print(f" Factor de inflación de SE (√DEFF): {inflation_factor:.2f}x")
print(f" Tamaño muestral efectivo (n_eff): {n_eff:,.0f} hogares equivalentes MAS")
print()
print("Consecuencia Metodológica:")
print(f" Los errores estándar calculados bajo supuestos MAS están subestimados en un {(inflation_factor-1)*100:.1f}%.")
print(" Sin embargo, debido a que n_eff = 22,942 sigue siendo masivo, todos los coeficientes clave")
print(" (OR educación superior, ruralidad, situación económica) mantienen p < 0.0001 (alta significancia).")

## 5. Heterogeneidad en Tamaños Muestrales Territoriales

In [ ]:
dpto_sample = df_h.groupby("departamento").agg(
 n_muestral=("id_hogar", "count"),
 hogares_expandidos=("fex_c", "sum"),
 pct_computador=("tiene_computador", lambda x: (x * df_h.loc[x.index, "fex_c"]).sum() / df_h.loc[x.index, "fex_c"].sum() * 100)
).sort_values("n_muestral", ascending=False)

print("TAMAÑOS MUESTRALES POR DEPARTAMENTO (EXTREMOS)")
print("=" * 75)
print("Top 5 con mayor muestra:")
for _, r in dpto_sample.head(5).iterrows():
 print(f" {r.name:<25} n = {int(r['n_muestral']):>5,} | Expandido: {int(r['hogares_expandidos']):>9,} | % Comp: {r['pct_computador']:.1f}%")

print("\nBottom 5 con menor muestra:")
for _, r in dpto_sample.tail(5).iterrows():
 print(f" {r.name:<25} n = {int(r['n_muestral']):>5,} | Expandido: {int(r['hogares_expandidos']):>9,} | % Comp: {r['pct_computador']:.1f}%")

print()
print(f"Razón de tamaño muestral (Máx / Mín): {dpto_sample['n_muestral'].max() / dpto_sample['n_muestral'].min():.1f}x")

## 6. Matriz de Síntesis: Limitaciones, Impacto y Estrategias de Mitigación

In [ ]:
matriz = pd.DataFrame([
 {
 "Limitación": "1. Corte transversal (Cross-sectional)",
 "Severidad": "Media",
 "Riesgo Principal": "Imposibilidad de probar causalidad estricta",
 "Mitigación Aplicada": "Interpretación probabilística (Odds Ratios) con múltiples controles de confusión"
 },
 {
 "Limitación": "2. Ausencia de ingreso continuo",
 "Severidad": "Media-Alta",
 "Riesgo Principal": "Sesgo de variable omitida en educación",
 "Mitigación Aplicada": "Uso conjunto de percepción (P3353), pobreza multidimensional (IAMG) y probabilidad P1911"
 },
 {
 "Limitación": "3. Métrica binaria de acceso",
 "Severidad": "Media",
 "Riesgo Principal": "Enmascaramiento de brechas de calidad/velocidad",
 "Mitigación Aplicada": "Desagregación entre conexión móvil, banda ancha fija y tenencia de computador"
 },
 {
 "Limitación": "4. Efecto de diseño muestral (DEFF=3.72)",
 "Severidad": "Baja-Media",
 "Riesgo Principal": "Subestimación de errores estándar bajo MAS",
 "Mitigación Aplicada": "Estimadores de Horvitz-Thompson con FEX_C y verificación de n_efectivo masivo (22,942)"
 },
 {
 "Limitación": "5. Falacia ecológica territorial",
 "Severidad": "Baja",
 "Riesgo Principal": "Extrapolación errónea del nivel depto al individuo",
 "Mitigación Aplicada": "Modelado paralelo en dos micro-niveles diferenciados: Hogar (85k) y Persona (214k)"
 },
 {
 "Limitación": "6. Varianza en periferia territorial",
 "Severidad": "Media",
 "Riesgo Principal": "Mayor dispersión muestral en Amazonía/Orinoquía",
 "Mitigación Aplicada": "Agrupación analítica complementaria por 6 macro-regiones DANE"
 }
])

print("MATRIZ METODOLÓGICA DE LIMITACIONES Y MITIGACIONES")
print("=" * 105)
for _, row in matriz.iterrows():
 print(f"\n{row['Limitación']} [Severidad: {row['Severidad']}]")
 print(f" Riesgo: {row['Riesgo Principal']}")
 print(f" Mitigación: {row['Mitigación Aplicada']}")

## 7. Conclusiones Metodológicas